# urban-traffic-forecasting — Colab 데모 / Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urbsn4i-sw/urban-traffic-forecasting/blob/main/notebooks/demo.ipynb)

교통 시공간 예측 STGNN(METR-LA · PEMS-BAY)의 **파이프라인 시연** 노트북입니다.
데이터 취득 → 기준선 → STGNN 짧은 학습 시연 → **전체 학습 결과 확인** → 시각화(RQ1/RQ2, 두 도시).

> ⚠️ **정직성 안내 (중요):** 이 노트북의 STGNN 학습은 **2에폭·서브셋의 "시연"**입니다.
> **실제 성능 결과가 아닙니다.** 저장소 `results/`의 metrics.json(전체 학습, seed 고정)이 최종 결과이며,
> 아래 §5에서 불러와 시연값과 **명확히 구분**해 표시합니다. "결과를 지어내지 않는다"를 노트북에도 적용합니다.

**앵커→브리지 개념 대응:** GraphCast(Lam+ 2023, 지구 격자→그래프→롤아웃) ↔ 교통 STGNN
(도로 센서→그래프, 다단계 롤아웃). 상세: [README](https://github.com/urbsn4i-sw/urban-traffic-forecasting#anchor--bridge-graphcast--traffic-stgnn).

## 1. 환경 설정 (Colab)

저장소를 clone 하고 경량 의존성을 설치합니다. **런타임 → 런타임 유형 변경 → GPU** 권장(없으면 CPU 폴백).

In [ ]:
import os, sys
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # (윈도우/로컬 OpenMP 충돌 회피; Colab 무해)

# Colab: 저장소 clone (이미 repo 안이면 skip)
if not os.path.isdir("src") and not os.path.isdir("urban-traffic-forecasting"):
    os.system("git clone https://github.com/urbsn4i-sw/urban-traffic-forecasting.git")
if os.path.isdir("urban-traffic-forecasting"):
    os.chdir("urban-traffic-forecasting")

# 경량 의존성 (Colab 엔 torch/numpy/pandas 기본; 나머지만)
os.system("pip -q install h5py gdown matplotlib pyyaml")

sys.path.insert(0, os.getcwd()); sys.path.insert(0, os.path.join(os.getcwd(), "src"))
import numpy as np, pandas as pd  # noqa
import data as D, baselines as B, metrics as M, graph as G, model as Model  # noqa
from common.metrics import rollout_divergence  # noqa
print("cwd:", os.getcwd())
try:
    import torch; print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
except Exception as e:
    print("torch 미설치/미탐지:", e)

## 2. 데이터 — METR-LA 취득 (연구용 공개)

`gdown`으로 공식 DCRNN Google Drive에서 `metr-la.h5`(207센서·5분)를 받습니다.
**데이터는 세션 내에서만 사용하며 커밋하지 않습니다.**

In [ ]:
os.makedirs("data/sensor_graph", exist_ok=True)
# metr-la.h5 (공식 DCRNN Drive 단일 파일 id)
if not os.path.exists("data/metr-la.h5"):
    import gdown
    gdown.download(id="1pAGRfzMx6K9WWsfDcD1NMbIif0T0saFC", output="data/metr-la.h5", quiet=True)
# 센서 거리 그래프(도로망 인접행렬용)
if not os.path.exists("data/sensor_graph/distances_la_2012.csv"):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/distances_la_2012.csv",
        "data/sensor_graph/distances_la_2012.csv")

series = D.load_h5_traffic("data/metr-la.h5")   # (T, N)
T, N = series.shape
print(f"METR-LA (T,N)=({T},{N}) — 센서 {N}개, 5분 간격, 결측(=0) {np.mean(series==0)*100:.2f}%")
assert N == 207, "센서 수가 207이 아니면 데이터 확인 필요"

## 3. 기준선 — copy-last · seasonal-HA (실제 측정값)

**전체 데이터**로 즉시 측정하므로 이 값들은 저장소 `results/`의 기준선과 일치합니다(결정론적).

In [ ]:
T_IN, HORIZON, PERIOD, HZ = 12, 12, 7*24*12, (3, 6, 12)
W = T - T_IN - HORIZON + 1
n_tr, n_va, n_te = D.chronological_split_sizes(W, (0.7, 0.1, 0.2))
test_w = np.arange(n_tr + n_va, W); tstart = test_w + T_IN
gt = np.stack([series[tstart + h] for h in range(HORIZON)], axis=1)         # (n_te,H,N)
pred_copy = np.repeat(series[tstart - 1][:, None, :], HORIZON, axis=1)
table = B.seasonal_average_table(series, train_len=n_tr + T_IN, period=PERIOD, null_val=0.0)
pred_ha = B.seasonal_ha_predict(table, tstart, HORIZON, PERIOD)

def mae_at(pred):
    p, g = np.moveaxis(pred, 1, 0), np.moveaxis(gt, 1, 0)
    return M.metrics_at_horizons(p, g, horizons=HZ, null_val=0.0)["mae"]
print("test 윈도우:", len(test_w))
print("copy-last  MAE @15/30/60 =", {k: round(v, 3) for k, v in mae_at(pred_copy).items()})
print("seasonal-HA MAE @15/30/60 =", {k: round(v, 3) for k, v in mae_at(pred_ha).items()})

## 4. STGNN 시연 학습 — ⚠️ **2에폭·서브셋 (시연용, 전체 결과 아님)**

파이프라인이 도는지 **원리 시연**만 합니다. 몇 에폭·서브셋이라 성능은 낮습니다.
**최종 성능은 §5의 전체 학습 결과를 보세요.** (GPU 권장; 없으면 CPU로 더 느리게 동작.)

In [ ]:
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
df = pd.read_hdf("data/metr-la.h5"); idx = pd.DatetimeIndex(df.index)
tod = ((idx.hour*3600 + idx.minute*60 + idx.second)/86400.0).to_numpy()
sc = D.Scaler.fit(series[:n_tr+T_IN], null_val=0.0)
feat = np.stack([sc.transform(series), np.repeat(tod[:, None], N, axis=1)], axis=-1).astype(np.float32)
Dmx = D.build_distance_matrix("data/sensor_graph/distances_la_2012.csv",
                              [str(c) for c in df.columns], has_header=True)
adj = G.normalize_adj_random_walk(G.gaussian_kernel_adjacency(Dmx, threshold=0.1))
feat_t = torch.tensor(feat, device=dev); speed_t = torch.tensor(series, dtype=torch.float32, device=dev)
adj_t = torch.tensor(adj, dtype=torch.float32, device=dev)

net = Model.build_model(num_nodes=N, in_dim=2, out_dim=1, horizon=HORIZON, hidden=32,
                        n_layers=2, diffusion_order=2, adj_mode="learned", adj_fixed=adj_t).to(dev)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
DEMO_WINDOWS = min(n_tr, 3000)          # 서브셋(빠른 시연). 전체는 결과가 §5.
demo_starts = np.arange(0, DEMO_WINDOWS)
print(f"[demo] device={dev}, 서브셋 {DEMO_WINDOWS} 윈도우, 2 epochs — 시연용")
for ep in range(2):
    net.train(); order = np.random.default_rng(ep).permutation(demo_starts)
    for i in range(0, len(order), 256):
        s = torch.as_tensor(order[i:i+256], dtype=torch.long, device=dev)
        ix = s[:, None] + torch.arange(T_IN, device=dev)[None, :]
        iy = s[:, None] + T_IN + torch.arange(HORIZON, device=dev)[None, :]
        x, y = feat_t[ix], speed_t[iy]
        pred = net(x)[..., 0]*sc.std + sc.mean
        mask = (y != 0).float(); loss = (torch.abs(pred - y)*mask).sum()/mask.sum().clamp_min(1)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"[demo] epoch {ep+1}/2  train_mae={loss.item():.3f}  ⚠️ 시연값(전체 결과 아님)")

## 5. 전체 학습 결과 로드 — `results/*.json` (최종 결과)

저장소에 커밋된 **전체 학습**(seed 고정, ≤50 epochs) 결과입니다. **§4 시연값과 다릅니다 — 이게 최종 결과입니다.**

In [ ]:
import glob, io, json
def load_full(city):
    f = sorted(glob.glob(f"results/stgnn-{city}-*/metrics.json"))[-1]
    return json.load(io.open(f, encoding="utf-8"))
full = {c: load_full(c) for c in ("metr-la", "pems-bay")}

print("=== 전체 학습 test MAE @15/30/60 (최종 결과, 시연값 아님) ===")
for city, d in full.items():
    print(f"\n[{d['dataset']['name']}]  (T,N)={d['dataset']['shape_T_N']}")
    for m in ("copy_last", "seasonal_historical_average"):
        mae = d["baselines_test"][m]["at_horizons"]["mae"]
        print(f"  {m:26s} {mae['h3']:.3f} / {mae['h6']:.3f} / {mae['h12']:.3f}")
    for m in ("fixed", "learned", "hybrid", "identity"):
        mae = d["stgnn_test"][m]["at_horizons"]["mae"]
        print(f"  STGNN {m:20s} {mae['h3']:.3f} / {mae['h6']:.3f} / {mae['h12']:.3f}")

## 6. 시각화

### 6-a. 예측 vs 실측 (한 센서) — 시연 모델
§4 시연 모델(few epochs)의 한 센서 예측을 실측·copy-last 와 비교. **시연이라 정확도는 낮습니다.**

In [ ]:
import matplotlib.pyplot as plt
net.eval()
with torch.no_grad():
    w0 = int(test_w[len(test_w)//2]); s0 = w0 + T_IN
    x0 = feat_t[torch.arange(w0, w0+T_IN, device=dev)][None]
    demo_pred = (net(x0)[0, :, :, 0]*sc.std + sc.mean).cpu().numpy()   # (H,N)
gt0 = series[s0:s0+HORIZON]; copy0 = np.repeat(series[s0-1][None], HORIZON, axis=0)
sensor = int(np.argmax(series[s0-1])); xs = np.arange(1, HORIZON+1)*5
plt.figure(figsize=(6.5, 3.2))
plt.plot(xs, gt0[:, sensor], "o-", label="actual")
plt.plot(xs, demo_pred[:, sensor], "s--", label="demo STGNN (few epochs)")
plt.plot(xs, copy0[:, sensor], "^:", label="copy-last")
plt.xlabel("minutes ahead"); plt.ylabel("speed (mph)")
plt.title(f"sensor {sensor}: forecast (⚠️ demo model)"); plt.legend(); plt.tight_layout(); plt.show()

### 6-b. RQ2 — 다단계 오차 누적 (전체 결과)
지평이 길어질수록 copy-last 는 오차가 빠르게 커지고, STGNN(learned)은 더 천천히 누적합니다.

In [ ]:
d = full["metr-la"]
cl = d["baselines_test"]["copy_last"]["per_step"]["per_step"]["mae"]
lr = d["stgnn_test"]["learned"]["per_step"]["per_step"]["mae"]
mins = np.arange(1, 13)*5
plt.figure(figsize=(6.5, 3.2))
plt.plot(mins, cl, "o-", label="copy-last")
plt.plot(mins, lr, "s-", label="STGNN learned (full)")
plt.xlabel("horizon (min)"); plt.ylabel("MAE"); plt.title("RQ2: error accumulation (METR-LA, full training)")
plt.legend(); plt.tight_layout(); plt.show()

### 6-c. RQ1 — 인접행렬 모드 비교 (전체 결과)
`learned ≈ hybrid` < `fixed`(도로망) < `identity`(그래프 없음). 학습 인접행렬이 이득이고, 그래프가 도움됩니다.

In [ ]:
modes = ["fixed", "learned", "hybrid", "identity"]
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2), sharey=False)
for k, city in enumerate(("metr-la", "pems-bay")):
    d = full[city]; vals = [d["stgnn_test"][m]["at_horizons"]["mae"]["h12"] for m in modes]
    colors = ["#999", "#2c7a7b", "#38a169", "#ccc"]
    ax[k].bar(modes, vals, color=colors)
    ax[k].set_title(f"{d['dataset']['name']} — MAE@60min"); ax[k].set_ylabel("MAE@60min")
    for i, v in enumerate(vals): ax[k].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
plt.suptitle("RQ1: adjacency modes (full training)"); plt.tight_layout(); plt.show()

### 6-d. 두 도시 비교 (전체 결과)
PEMS-BAY(고속도로·결측~0)는 오차가 METR-LA 보다 훨씬 낮습니다. 두 도시 모두 STGNN이 copy-last 를 이깁니다.

In [ ]:
cities = ["metr-la", "pems-bay"]; names = [full[c]["dataset"]["name"] for c in cities]
learned = [full[c]["stgnn_test"]["learned"]["at_horizons"]["mae"]["h12"] for c in cities]
copyl = [full[c]["baselines_test"]["copy_last"]["at_horizons"]["mae"]["h12"] for c in cities]
xx = np.arange(2); w = 0.35
plt.figure(figsize=(6.5, 3.2))
plt.bar(xx - w/2, copyl, w, label="copy-last")
plt.bar(xx + w/2, learned, w, label="STGNN learned")
plt.xticks(xx, names); plt.ylabel("MAE@60min"); plt.title("Two cities: MAE@60min (full training)")
plt.legend(); plt.tight_layout(); plt.show()

## 7. 마무리 / 한계

- **SOTA 아님(원리 재현):** 소형 2-layer STGNN 이라 논문값(METR-LA DCRNN 2.77/3.15/3.60)에는 못 미칩니다.
  목적은 RQ1(인접행렬 학습 이득)·RQ2(오차 누적)를 두 도시에서 실측 확인하는 것입니다.
- **§4 시연 학습(2에폭·서브셋)은 성능이 아닙니다.** 최종 결과는 §5의 전체 학습 metrics.json 입니다.
- **정직한 차이:** METR-LA 60분 지평에선 seasonal-HA 가 근소 우위; PEMS-BAY 에선 STGNN 이 전 지평 우세.
- 데이터·가중치는 저장소에 없습니다(연구용 공개, 세션 내 취득).

전체 설명·결과표·재현 명령: **[README](https://github.com/urbsn4i-sw/urban-traffic-forecasting)** · **[한국어 README](https://github.com/urbsn4i-sw/urban-traffic-forecasting/blob/main/README.ko.md)**